# FastAPI backend — Smoke notebook

Vérification manuelle du backend `api/`. Regroupe les tests à faire après chaque PR R4 qui touche à l'API REST ou WebSocket.

**Comment utiliser :**

1. Dans un shell séparé : `docker compose -f docker-compose.dev.yml up -d postgres redis` puis `bash scripts/run_api.sh` (ou `uvicorn api.main:app --reload`).
2. Ici : `Kernel > Restart & Run All`.
3. Avant chaque section, lis la cellule markdown **"Ce que tu dois tester"**.

**Prérequis une seule fois :**
- venv avec `fastapi`, `uvicorn`, `httpx` installés
- Postgres + Redis up

## 0. Setup — client httpx partagé

### Ce que tu dois tester
Le client peut joindre uvicorn sur `localhost:8000`.

**Succès attendu** : `root OK : 200` affiché (via /openapi.json). Pas d'exception.

**Alerte si tu vois** : `ConnectionRefused` → uvicorn n'est pas lancé, run `bash scripts/run_api.sh`.

In [ ]:
import httpx

BASE = "http://localhost:8000"
c = httpx.Client(base_url=BASE, timeout=5.0)

r = c.get("/openapi.json")
print("root OK :", r.status_code)
print("title   :", r.json()["info"]["title"], r.json()["info"]["version"])

## 1. Scaffold — `/docs`, `/openapi.json`, `/metrics` (R4 PR #1)

### Ce que tu dois tester
Les endpoints generic FastAPI répondent + `/metrics` renvoie du Prometheus text.

**Succès attendu :**
- `/docs` → 200, HTML Swagger UI
- `/openapi.json` → 200, JSON avec `info.title == "FXVol API"`
- `/metrics` → 200, contient `fxvol_http_request_duration_seconds`

**Alerte si tu vois :**
- 404 sur `/docs` → uvicorn pas lancé avec `api.main:app`
- `fxvol_*` absent de `/metrics` → le TimingMiddleware n'est pas wired

In [ ]:
for path in ("/docs", "/openapi.json", "/metrics"):
    r = c.get(path)
    print(f"{path:<20} → {r.status_code}  ({len(r.content)} bytes)")

In [ ]:
# Le /metrics doit contenir le name de notre histogram.
metrics = c.get("/metrics").text
assert "fxvol_http_request_duration_seconds" in metrics, "TimingMiddleware non wired"
print("OK — TimingMiddleware actif")

## 2. Health endpoints — `/api/v1/health` + `/extended` (R4 PR #2)

### Ce que tu dois tester
Liveness simple + readiness avec check Redis + DB + heartbeats des 3 engines.

**Succès attendu :**
- `/api/v1/health` → `{"status": "OK"}` toujours
- `/api/v1/health/extended` → JSON avec `components.redis`, `components.database`, `components.engines.{market_data,vol_engine,risk_engine}`
- Si `app.py` tourne en parallèle et produit des heartbeats → `engines.*` == `"OK"`
- Sans `app.py` (donc pas de heartbeats) → `engines.*` == `"DOWN"` et status global == `"DEGRADED"`

**Alerte si tu vois :**
- `redis: "DOWN"` → Redis n'est pas joignable sur `localhost:6380`
- `database: "DOWN"` → Postgres n'est pas up ou `DATABASE_URL` pointe au mauvais endroit
- 500 Internal Server Error → bug dans le router, regarder les logs uvicorn

In [ ]:
r = c.get("/api/v1/health")
print(r.status_code, r.json())
assert r.json() == {"status": "OK"}

In [ ]:
import json

r = c.get("/api/v1/health/extended")
print(r.status_code)
print(json.dumps(r.json(), indent=2))

# Le status global reflete l'aggregate
body = r.json()
print("\noverall :", body["status"])
for name, state in body["components"].items():
    print(f"  {name:<12} : {state}")

### Scénario : app.py produit des heartbeats → engines OK

Lance `app.py` en parallèle (shell séparé), clique Start Engine, attends 2-3s, re-run la cellule `/api/v1/health/extended` ci-dessus. Les 3 engines doivent passer à `"OK"`.

## 3. Cleanup

### Ce que tu dois tester
Rien — le client httpx s'arrête proprement.

In [ ]:
c.close()
print("client httpx fermé")

## Cheat sheet — troubleshooting

| Erreur | Cause probable | Fix |
|---|---|---|
| `ConnectionError` sur localhost:8000 | uvicorn pas lancé | `bash scripts/run_api.sh` |
| `No module named 'httpx'` | httpx pas dans venv | `python -m pip install httpx` |
| `redis: "DOWN"` | Redis pas up | `docker compose ... up -d redis` |
| `database: "DOWN"` | Postgres pas up ou mauvais URL | `docker compose ... up -d postgres` + check `$env:DATABASE_URL` |
| `engines.*: "DOWN"` | `app.py` ne tourne pas OU n'écrit pas les heartbeats | lancer `app.py` + Connect + Start Engine |
| `engines.*: "STALE (>30s)"` | engine a tourné puis a stoppé | c'est le bon comportement, re-clic Start Engine pour repush |